<a href="https://colab.research.google.com/github/Saadiphone/thalf/blob/main/TRELLIS_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# TRELLIS — متوافق مع Colab Python 3.12 + CUDA 12.4
# ============================================================
import subprocess, os, sys

def run(cmd):
    print(f"  > {cmd[:90]}")
    subprocess.run(cmd, shell=True)

# 1) تثبيت الحزم المتوافقة مع Python 3.12
print("[1/4] installing packages...")
run("apt install aria2 -qqy")
run("pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124")
run("pip install -q xformers")

os.chdir("/content")
if not os.path.exists("/content/TRELLIS/trellis"):
    run("git clone --recursive https://github.com/Microsoft/TRELLIS /content/TRELLIS")
os.chdir("/content/TRELLIS")

run("pip install -q easydict rembg onnxruntime onnxruntime-gpu plyfile huggingface-hub safetensors")
run("pip install -q 'pillow<12' 'numpy<2.1'")
run("pip install -q git+https://github.com/NVlabs/nvdiffrast")
run("pip install -q trimesh xatlas pyvista pymeshfix igraph")
run("pip install -q opencv-python imageio imageio-ffmpeg")
run("pip install -q spconv-cu124")

# kaolin — build from source for Python 3.12
print("  building kaolin (2-3 min)...")
run("pip install -q kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.6.0_cu124.html 2>/dev/null || pip install -q kaolin")

# diff-gaussian-rasterization — build from source
if not os.path.exists("/tmp/mip-splatting"):
    run("git clone https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting")
run("pip install -q --no-build-isolation /tmp/mip-splatting/submodules/diff-gaussian-rasterization/")

# utils3d
run("pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8")

print("  done")

# 2) تحميل الموديل
print("[2/4] downloading model...")
M = "/content/model/ckpts"
os.makedirs(M, exist_ok=True)
base = "https://huggingface.co/JeffreyXiang/TRELLIS-image-large"
files = {
    f"{base}/raw/main/pipeline.json": "/content/model/pipeline.json",
    f"{base}/raw/main/ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.json": f"{M}/slat_dec_gs_swin8_B_64l8gs32_fp16.json",
    f"{base}/resolve/main/ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.safetensors": f"{M}/slat_dec_gs_swin8_B_64l8gs32_fp16.safetensors",
    f"{base}/raw/main/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.json": f"{M}/slat_dec_mesh_swin8_B_64l8m256c_fp16.json",
    f"{base}/resolve/main/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.safetensors": f"{M}/slat_dec_mesh_swin8_B_64l8m256c_fp16.safetensors",
    f"{base}/raw/main/ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.json": f"{M}/slat_dec_rf_swin8_B_64l8r16_fp16.json",
    f"{base}/resolve/main/ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.safetensors": f"{M}/slat_dec_rf_swin8_B_64l8r16_fp16.safetensors",
    f"{base}/raw/main/ckpts/slat_enc_swin8_B_64l8_fp16.json": f"{M}/slat_enc_swin8_B_64l8_fp16.json",
    f"{base}/resolve/main/ckpts/slat_enc_swin8_B_64l8_fp16.safetensors": f"{M}/slat_enc_swin8_B_64l8_fp16.safetensors",
    f"{base}/raw/main/ckpts/slat_flow_img_dit_L_64l8p2_fp16.json": f"{M}/slat_flow_img_dit_L_64l8p2_fp16.json",
    f"{base}/resolve/main/ckpts/slat_flow_img_dit_L_64l8p2_fp16.safetensors": f"{M}/slat_flow_img_dit_L_64l8p2_fp16.safetensors",
    f"{base}/raw/main/ckpts/ss_dec_conv3d_16l8_fp16.json": f"{M}/ss_dec_conv3d_16l8_fp16.json",
    f"{base}/resolve/main/ckpts/ss_dec_conv3d_16l8_fp16.safetensors": f"{M}/ss_dec_conv3d_16l8_fp16.safetensors",
    f"{base}/raw/main/ckpts/ss_enc_conv3d_16l8_fp16.json": f"{M}/ss_enc_conv3d_16l8_fp16.json",
    f"{base}/resolve/main/ckpts/ss_enc_conv3d_16l8_fp16.safetensors": f"{M}/ss_enc_conv3d_16l8_fp16.safetensors",
    f"{base}/raw/main/ckpts/ss_flow_img_dit_L_16l8_fp16.json": f"{M}/ss_flow_img_dit_L_16l8_fp16.json",
    f"{base}/resolve/main/ckpts/ss_flow_img_dit_L_16l8_fp16.safetensors": f"{M}/ss_flow_img_dit_L_16l8_fp16.safetensors",
}
for url, path in files.items():
    if not os.path.exists(path):
        d = os.path.dirname(path)
        f = os.path.basename(path)
        run(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{d}' -o '{f}'")

# DINOv2
run("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/facebookresearch/dinov2/zipball/main -d /root/.cache/torch/hub -o main.zip")
run("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth -d /root/.cache/torch/hub/checkpoints -o dinov2_vitl14_reg4_pretrain.pth")
run("aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx -d /root/.u2net -o u2net.onnx")
print("  done")

# 3) Patch
print("[3/4] patching...")
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# spconv backend
init_f = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_f) as f: c = f.read()
with open(init_f, 'w') as f: f.write(c.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'"))

# xformers attention patch
attn_f = "/content/TRELLIS/trellis/modules/sparse/attention/full_attn.py"
with open(attn_f) as f: ac = f.read()
xp = "\ntry:\n    from xformers.ops.fmha.attn_bias import BlockDiagonalMask as _BDM\n    import xformers.ops.fmha as _fmha\n    if not hasattr(_fmha, 'BlockDiagonalMask'): _fmha.BlockDiagonalMask = _BDM\nexcept: pass\n"
if 'BlockDiagonalMask' not in ac:
    with open(attn_f, 'w') as f: f.write(xp + ac)

# flexicubes kaolin fix
fc = "/content/TRELLIS/trellis/representations/mesh/flexicubes/flexicubes.py"
if os.path.exists(fc):
    with open(fc) as f: txt = f.read()
    if 'from kaolin' in txt:
        txt = txt.replace(
            "from kaolin.utils.testing import check_tensor",
            "def check_tensor(tensor, shape=None, throw=True): return True"
        )
        with open(fc, 'w') as f: f.write(txt)

os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
print("  done")

# 4) Load & Run
print("[4/4] loading model...")
sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

import numpy as np, torch, imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ TRELLIS ready! run next cell to generate 3D")

[1/4] installing packages...
  > apt install aria2 -qqy
  > pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org
  > pip install -q xformers
  > pip install -q easydict rembg onnxruntime onnxruntime-gpu plyfile huggingface-hub safetens
  > pip install -q 'pillow<12' 'numpy<2.1'
  > pip install -q git+https://github.com/NVlabs/nvdiffrast
  > pip install -q trimesh xatlas pyvista pymeshfix igraph
  > pip install -q opencv-python imageio imageio-ffmpeg
  > pip install -q spconv-cu124
  building kaolin (2-3 min)...
  > pip install -q kaolin -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.6.0_cu12
  > git clone https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
  > pip install -q --no-build-isolation /tmp/mip-splatting/submodules/diff-gaussian-rasterizat
  > pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460
  done
[2/4] downloading model...
  > aria2c --console-log-level=error -c

SyntaxError: invalid syntax (flexicubes.py, line 59)

In [10]:
import subprocess, sys, os

# 1) ثبّت nvdiffrast من wheel جاهز
print("installing nvdiffrast...")
r = subprocess.run("pip install nvdiffrast -f https://nvdiffrast.s3.us-east-2.amazonaws.com/whl/torch-2.6.0_cu124.html 2>&1 | tail -5", shell=True, capture_output=True, text=True)
print(r.stdout)

# لو ما اشتغل، نجرب من PyPI مباشرة
try:
    import nvdiffrast
    print("✅ nvdiffrast OK!")
except:
    print("trying alternative...")
    subprocess.run("pip install nvdiffrast", shell=True)
    try:
        import nvdiffrast
        print("✅ nvdiffrast OK!")
    except:
        # آخر حل — نثبّت ninja أولاً ونبني من المصدر
        print("building from source with ninja...")
        subprocess.run("pip install ninja", shell=True)
        subprocess.run("pip install --no-build-isolation git+https://github.com/NVlabs/nvdiffrast", shell=True)
        try:
            import nvdiffrast
            print("✅ nvdiffrast OK!")
        except:
            print("❌ trying pre-built wheel from HF...")
            # نستخدم نفس الـ wheel من HuggingFace اللي camenduru يستخدمه بس نبني لـ 3.12
            subprocess.run("pip install https://huggingface.co/spaces/JeffreyXiang/TRELLIS/resolve/main/wheels/nvdiffrast-0.3.3-cp310-cp310-linux_x86_64.whl 2>&1 | tail -3", shell=True)
            # لو ما اشتغل لأنه cp310، نبني بطريقة ثانية
            subprocess.run("""
cd /tmp && rm -rf nvdiffrast_build && git clone https://github.com/NVlabs/nvdiffrast nvdiffrast_build && \
cd nvdiffrast_build && pip install --no-build-isolation -e . 2>&1 | tail -10
""", shell=True)

# تحقق نهائي
import importlib
try:
    nvdiffrast = importlib.import_module('nvdiffrast')
    print("\n✅✅ nvdiffrast installed!")
except:
    # نشوف وش مثبّت
    r2 = subprocess.run("pip list | grep -i nvdiff", shell=True, capture_output=True, text=True)
    print(f"installed: {r2.stdout}")
    r3 = subprocess.run("pip list | grep -i rast", shell=True, capture_output=True, text=True)
    print(f"installed: {r3.stdout}")
    # نجرب import بطريقة ثانية
    try:
        import nvdiffrast.torch as dr
        print("✅ nvdiffrast.torch works!")
    except Exception as e:
        print(f"❌ final error: {e}")

installing nvdiffrast...
Looking in links: https://nvdiffrast.s3.us-east-2.amazonaws.com/whl/torch-2.6.0_cu124.html

trying alternative...
building from source with ninja...
✅ nvdiffrast OK!

✅✅ nvdiffrast installed!


In [11]:
import os, sys

os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
mods = [k for k in sys.modules if k.startswith('trellis')]
for m in mods: del sys.modules[m]

sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

import torch, numpy as np, imageio
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils
print("✅ all imports OK!")

print("loading model...")
with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ TRELLIS READY!")

[SPARSE] Backend: spconv, Attention: xformers
✅ all imports OK!
loading model...
[SPARSE][CONV] spconv algo: native


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[ATTENTION] Using backend: xformers


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✅ TRELLIS READY!


In [16]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
used = torch.cuda.memory_allocated()/1024**3
total = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"GPU memory: {used:.1f}GB / {total:.1f}GB")
print("✅ memory cleared!")

GPU memory: 5.4GB / 14.6GB
✅ memory cleared!


In [18]:
# نمسح كل شي ونعيد تحميل صح
import torch, gc
gc.collect()
torch.cuda.empty_cache()

# نمسح الـ pipeline القديم
try: del pipeline
except: pass
gc.collect()
torch.cuda.empty_cache()

# نحمّل بدون inference_mode
print("reloading model...")
with torch.no_grad():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ model loaded!")

# Gradio app
import numpy as np, imageio, gradio as gr
from PIL import Image

def generate_3d(image, seed, steps):
    if image is None:
        return None, None, "ارفع صورة أولاً"
    try:
        gc.collect()
        torch.cuda.empty_cache()

        img = Image.fromarray(image).convert("RGBA")
        processed = pipeline.preprocess_image(img)

        with torch.no_grad():
            outputs = pipeline.run(processed, seed=int(seed),
                formats=["gaussian", "mesh"],
                preprocess_image=False,
                sparse_structure_sampler_params={"steps": int(steps), "cfg_strength": 7.5},
                slat_sampler_params={"steps": int(steps), "cfg_strength": 3.0})

            video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
            video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
            video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
            imageio.mimsave("/content/output.mp4", video, fps=15)

        # GLB needs grad
        glb = postprocessing_utils.to_glb(outputs['gaussian'][0], outputs['mesh'][0],
            simplify=0.95, texture_size=1024, verbose=False)
        glb.export("/content/output.glb")

        del outputs, video, video_geo
        gc.collect()
        torch.cuda.empty_cache()

        return "/content/output.mp4", "/content/output.glb", f"✅ تم! Seed: {seed}"
    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        return None, None, f"❌ {str(e)}"

with gr.Blocks(title="TRELLIS 3D") as demo:
    gr.Markdown("# 🎨 TRELLIS - Image to 3D")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="ارفع صورة")
            seed = gr.Number(label="Seed", value=42)
            steps = gr.Slider(6, 20, value=12, step=1, label="Steps")
            btn = gr.Button("🚀 Generate 3D", variant="primary")
        with gr.Column():
            video_out = gr.Video(label="Preview")
            glb_out = gr.File(label="Download GLB")
            status = gr.Textbox(label="Status")
    btn.click(generate_3d, [input_image, seed, steps], [video_out, glb_out, status])

demo.launch(share=True)

reloading model...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


✅ model loaded!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c8dea15d06b7ce94d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
import torch, gc, numpy as np, imageio
from PIL import Image
from google.colab import files

gc.collect()
torch.cuda.empty_cache()

# ارفع صورتك
print("ارفع صورة...")
uploaded = files.upload()
input_image = list(uploaded.keys())[0]

img = Image.open(input_image).convert("RGBA")
processed = pipeline.preprocess_image(img)

print("generating...")
with torch.no_grad():
    outputs = pipeline.run(processed, seed=42, formats=["gaussian", "mesh"],
        preprocess_image=False,
        sparse_structure_sampler_params={"steps": 12, "cfg_strength": 7.5},
        slat_sampler_params={"steps": 12, "cfg_strength": 3.0})

print("rendering video...")
with torch.no_grad():
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=60)['color']
    imageio.mimsave("/content/output.mp4", video, fps=15)
print("✅ video saved!")

print("exporting GLB...")
# نسوي clone للـ tensors عشان تشتغل مع autograd
gs = outputs['gaussian'][0]
gs._xyz = gs._xyz.clone().detach()
gs._features_dc = gs._features_dc.clone().detach()
gs._scaling = gs._scaling.clone().detach()
gs._rotation = gs._rotation.clone().detach()
gs._opacity = gs._opacity.clone().detach()

mesh = outputs['mesh'][0]
mesh.vertices = mesh.vertices.clone().detach()
mesh.faces = mesh.faces.clone().detach()

glb = postprocessing_utils.to_glb(gs, mesh,
    simplify=0.95, texture_size=1024, verbose=False)
glb.export("/content/output.glb")
print("✅ GLB saved!")

files.download("/content/output.glb")
files.download("/content/output.mp4")

ارفع صورة...


Saving ‏لقطة الشاشة ١٤٤٧-٠٩-٢٩ في ٧.٠٦.٣٠ م.png to ‏لقطة الشاشة ١٤٤٧-٠٩-٢٩ في ٧.٠٦.٣٠ م (3).png
generating...


Sampling: 100%|██████████| 12/12 [00:07<00:00,  1.57it/s]


rendering video...


Rendering: 60it [00:00, 121.79it/s]


...

✅ GLB saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>